In [1]:
import pandas as pd

In [2]:
df_pay = pd.read_csv("data/03 Оплаты ХК.csv", sep=";")

In [3]:
df_pay.head()

,Номер,Дата оплаты,Сумма,Способ оплаты
0,15,24.01.2025,100,5.0
1,15,25.01.2026,1240,5.0
2,15,25.05.2025,"293,96",5.0
3,15,25.06.2025,10,5.0
4,15,26.02.2025,"289,56",5.0


In [4]:
df_pay.groupby(by="Номер",as_index=False).agg({"Сумма": "count"})

,Номер,Сумма
0,15,16
1,16,15
2,17,13
3,18,15
4,20,13
...,...,...
128717,147099,16
128718,147100,14
128719,147101,15
128720,147102,15


In [ ]:
from form_time_features import extract_payment_features

payment_features = extract_payment_features(k=3, current_date=pd.to_datetime('2026-04-23'))
print(df_pay['Номер'].unique().shape, payment_features.shape)
print(f"Число неплатёжников: {(payment_features['Платежей_за_последние_3_мес'] == 0).sum()}")
payment_features.head()

In [ ]:
from form_time_features import calculate_complex_features

complex_features = calculate_complex_features(k=3, curr_date=pd.to_datetime('2026-04-23'))
complex_features.head()

,Id,Days_Since_Clearance,Payment_Fraction_12M,Consecutive_Debt_Months,Payment_Accrual_Ratio_kM,Balance_Trend_Slope_3M,Debt_to_Avg_Accrual_3M,Days_Since_Advance_5th,Days_Since_Salary_20th
0,1.0,9999.0,0.0,0,1.0,0.0,0.0,18,3
1,2.0,9999.0,0.0,0,1.0,0.0,0.0,18,3
2,3.0,9999.0,0.0,0,1.0,0.0,0.0,18,3
3,4.0,9999.0,0.0,0,1.0,0.0,0.0,18,3
4,5.0,9999.0,0.0,0,1.0,0.0,0.0,18,3


In [ ]:
complex_features_cut = complex_features[complex_features['Days_Since_Clearance'] != 9999]
print(complex_features_cut.shape)
complex_features_cut.head()

(54441, 9)


,Id,Days_Since_Clearance,Payment_Fraction_12M,Consecutive_Debt_Months,Payment_Accrual_Ratio_kM,Balance_Trend_Slope_3M,Debt_to_Avg_Accrual_3M,Days_Since_Advance_5th,Days_Since_Salary_20th
14,15.0,302.0,1.000000,9,1.403948,153.80,1.003642,18,3
15,16.0,36.0,1.000000,0,0.950364,223.72,-0.119746,18,3
19,20.0,101.0,0.818182,2,0.753012,-1659.24,0.443373,18,3
22,23.0,44.0,1.000000,0,0.756637,-1040.48,-0.032448,18,3
24,25.0,25.0,0.636364,0,1.463569,-4557.96,-0.496304,18,3


In [ ]:
# Проверка число сезонных признаков
from form_time_features import get_seasonality_features

current_date=pd.to_datetime('2026-04-23')
df_season = get_seasonality_features(current_date)
df_season.head()

ImportError: cannot import name 'get_seasonality_features' from 'form_time_features' (d:\Energo_Hack_Hackaton\form_time_features.py)

In [ ]:
import pandas as pd
import os

# путь к главному файлу
main_file = "data/14 Лимиты мер воздействия ХК.xlsx"

# читаем главный файл
limits_df = pd.read_excel(main_file)

# словарь для результатов
result = {}

for _, row in limits_df.iterrows():
    file_name = row.iloc[0]
    limit = row.iloc[1]

    # пропускаем пустые строки
    if pd.isna(file_name):
        continue

    file_path = os.path.join("data", file_name+".xlsx")

    # читаем файл без заголовков
    df_raw = pd.read_excel(file_path, header=None)

    # 1 строка — название операции
    operation_name = df_raw.iloc[0, 0]

    # 2 строка — заголовки
    df = df_raw.iloc[2:].copy()
    df.columns = df_raw.iloc[1]

    # чистка: убираем #Н/Д
    df = df[df["ЛС"].notna()]

    # сохраняем
    result[operation_name] = {
        "limit": limit,
        "data": df
    }

# теперь result — словарь с данными

In [ ]:
result.keys()

dict_keys(['Автодозвон', 'E-mail', 'СМС', 'Обзвон оператором', 'Претензия', 'Выезд к абоненту', 'Уведомление о введении ограничения', 'Ограничение', 'Заявление о выдаче судебного приказа', 'Получение судебного приказа или ИЛ'])

In [ ]:
result["Заявление о выдаче судебного приказа"]

{'limit': 400,
 'data': 1         ЛС                 Дата
 2         66  2025-07-30 00:00:00
 3         66  2025-07-30 00:00:00
 4         90  2025-11-24 00:00:00
 5         90  2025-11-24 00:00:00
 6        306  2025-09-25 00:00:00
 ...      ...                  ...
 8387  147017  2025-05-30 00:00:00
 8388  147017  2026-02-04 00:00:00
 8389  147017  2025-02-24 00:00:00
 8390  147017  2025-05-30 00:00:00
 8391  147017  2026-02-04 00:00:00
 
 [8390 rows x 2 columns]}

In [ ]:
df2 = pd.read_excel("data/01 Общая информация о ЛС ХК.xlsx", index_col=0)
df2.drop(columns=["Адрес (ГУИД)"], inplace=True)
df2

,Возможность дистанционного отключения,Наличие телефона,Наличие льгот,Газификация дома,Город,ЯрОблИЕРЦ квитанция,Почта России квитанция,электронная квитанция,не проживает,ЧД,МКД,Общежитие,Установка Тамбур,Установка опора,Установка в квартире/доме,Установка лестничкая клетка
ЛС,,,,,,,,,,,,,,,,
1,Нет,Нет,Нет,Да,Нет,Нет,Да,Нет,Нет,Нет,Да,Нет,Нет,Нет,Нет,Нет
2,Нет,Нет,Нет,Нет,Нет,Нет,Да,Нет,Нет,Нет,Да,Нет,Нет,Нет,Нет,Нет
3,Нет,Нет,Нет,Нет,Нет,Нет,Да,Нет,Нет,Нет,Да,Нет,Нет,Нет,Нет,Нет
4,Нет,Нет,Нет,Да,Нет,Нет,Да,Нет,Нет,Нет,Да,Нет,Нет,Нет,Нет,Да
5,Нет,Нет,Нет,Да,Нет,Нет,Да,Нет,Нет,Нет,Да,Нет,Нет,Нет,Нет,Нет
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
150817,Нет,Нет,Нет,Да,Да,Да,Нет,Нет,Нет,Нет,Да,Нет,Нет,Нет,Нет,Нет
150818,Нет,Нет,Нет,Да,Да,Да,Нет,Нет,Нет,Нет,Да,Нет,Нет,Нет,Нет,Нет
150819,Нет,Нет,Нет,Да,Да,Нет,Да,Нет,Нет,Нет,Да,Нет,Нет,Нет,Нет,Нет
